# 01 - Knowledge Base 설정

이 노트북에서는 Agentic Retrieval을 위한 **Knowledge Source**와 **Knowledge Base**를 생성합니다.

## 📋 진행 순서
1. 환경 설정 및 연결 확인
2. 벡터 및 시맨틱 검색을 지원하는 인덱스 생성
3. 샘플 데이터 업로드 (벡터 포함)
4. Knowledge Source 생성
5. Knowledge Base 생성

## 1. 환경 설정

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents import SearchClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
)
from openai import AzureOpenAI

# 환경 변수 로드
load_dotenv()

# Azure AI Search 설정
search_endpoint = os.environ["SEARCH_ENDPOINT"]
search_api_key = os.environ["SEARCH_ADMIN_KEY"]
search_credential = AzureKeyCredential(search_api_key)

# Azure OpenAI 설정
aoai_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
aoai_api_key = os.environ["FOUNDRY_PROJECT_KEY"]
api_version = os.environ.get("AZURE_OPENAI_API_VERSION", "2025-05-01-preview")
embedding_deployment = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
chat_deployment = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"]

# 인덱스 이름
INDEX_NAME = "products-agentic"
KNOWLEDGE_SOURCE_NAME = "products-source"
KNOWLEDGE_BASE_NAME = "products-kb"

print(f"✅ Search Endpoint: {search_endpoint}")
print(f"✅ OpenAI Endpoint: {aoai_endpoint}")
print(f"✅ Embedding Model: {embedding_deployment}")
print(f"✅ Chat Model: {chat_deployment}")

## 2. OpenAI 클라이언트 초기화

In [ ]:
# Azure OpenAI 클라이언트 생성
aoai_client = AzureOpenAI(
    azure_endpoint=aoai_endpoint,
    api_key=aoai_api_key,
    api_version=api_version
)

# 임베딩 함수 정의
def get_embedding(text: str) -> list[float]:
    """텍스트를 벡터로 변환합니다."""
    response = aoai_client.embeddings.create(
        input=text,
        model=embedding_deployment
    )
    return response.data[0].embedding

# 테스트
test_embedding = get_embedding("테스트 문장")
print(f"✅ Embedding 차원: {len(test_embedding)}")

## 3. 벡터 + 시맨틱 검색 인덱스 생성

Agentic Retrieval을 위해서는 **Semantic Ranker**가 활성화된 인덱스가 필요합니다.

In [ ]:
# 임베딩 차원 확인
EMBEDDING_DIMENSIONS = len(test_embedding)

# 인덱스 스키마 정의
index = SearchIndex(
    name=INDEX_NAME,
    fields=[
        SearchField(
            name="id",
            type=SearchFieldDataType.String,
            key=True,
            sortable=True,
            filterable=True
        ),
        SearchField(
            name="title",
            type=SearchFieldDataType.String,
            searchable=True,
            filterable=True,
            sortable=True
        ),
        SearchField(
            name="brand",
            type=SearchFieldDataType.String,
            searchable=True,
            filterable=True,
            facetable=True
        ),
        SearchField(
            name="category",
            type=SearchFieldDataType.String,
            searchable=True,
            filterable=True,
            facetable=True
        ),
        SearchField(
            name="price",
            type=SearchFieldDataType.Double,
            filterable=True,
            sortable=True
        ),
        SearchField(
            name="review",
            type=SearchFieldDataType.String,
            searchable=True
        ),
        SearchField(
            name="content",
            type=SearchFieldDataType.String,
            searchable=True,
            analyzer_name="ko.microsoft"
        ),
        SearchField(
            name="content_vector",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            searchable=True,
            vector_search_dimensions=EMBEDDING_DIMENSIONS,
            vector_search_profile_name="hnsw-profile"
        )
    ],
    # 벡터 검색 설정
    vector_search=VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(name="hnsw-config")
        ],
        profiles=[
            VectorSearchProfile(
                name="hnsw-profile",
                algorithm_configuration_name="hnsw-config"
            )
        ]
    ),
    # 시맨틱 검색 설정 (Agentic Retrieval 필수)
    semantic_search=SemanticSearch(
        configurations=[
            SemanticConfiguration(
                name="semantic-config",
                prioritized_fields=SemanticPrioritizedFields(
                    title_field=SemanticField(field_name="title"),
                    content_fields=[SemanticField(field_name="content")],
                    keywords_fields=[SemanticField(field_name="brand"), SemanticField(field_name="category")]
                )
            )
        ],
        default_configuration_name="semantic-config"
    )
)

# 인덱스 생성
index_client = SearchIndexClient(endpoint=search_endpoint, credential=search_credential)

# 기존 인덱스가 있으면 삭제
try:
    index_client.delete_index(INDEX_NAME)
    print(f"🗑️ 기존 인덱스 '{INDEX_NAME}' 삭제됨")
except:
    pass

result = index_client.create_index(index)
print(f"✅ 인덱스 '{result.name}' 생성 완료")

## 4. 샘플 데이터 로드 및 벡터화

In [ ]:
import csv
import time

# CSV 파일 읽기
data_path = "../00-data/sample_data.csv"
documents = []

with open(data_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # content 필드 생성 (제목 + 브랜드 + 카테고리 + 리뷰)
        content = f"{row['title']} - {row['brand']} - {row['category']}. {row['review']}"
        
        doc = {
            "id": row['id'],
            "title": row['title'],
            "brand": row['brand'],
            "category": row['category'],
            "price": float(row['normal_price']) if row['normal_price'] else 0.0,
            "review": row['review'],
            "content": content
        }
        documents.append(doc)

print(f"📄 로드된 문서 수: {len(documents)}")
print(f"📝 샘플 문서:")
print(documents[0])

In [ ]:
# 벡터화 (임베딩 생성)
print("🔄 임베딩 생성 중...")

# 배치 단위로 처리 (Rate Limit 방지)
batch_size = 20
for i in range(0, len(documents), batch_size):
    batch = documents[i:i+batch_size]
    for doc in batch:
        doc["content_vector"] = get_embedding(doc["content"])
    
    print(f"  진행률: {min(i+batch_size, len(documents))}/{len(documents)}")
    time.sleep(1)  # Rate Limit 방지

print(f"✅ 임베딩 생성 완료")

## 5. 데이터 업로드

In [ ]:
# 검색 클라이언트 생성
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=INDEX_NAME,
    credential=search_credential
)

# 문서 업로드 (배치 단위)
batch_size = 50
for i in range(0, len(documents), batch_size):
    batch = documents[i:i+batch_size]
    result = search_client.upload_documents(documents=batch)
    succeeded = sum(1 for r in result if r.succeeded)
    print(f"  배치 {i//batch_size + 1}: {succeeded}/{len(batch)} 문서 업로드 성공")

print(f"\n✅ 총 {len(documents)}개 문서 업로드 완료")

## 6. Knowledge Source 생성

**Knowledge Source**는 AI Search 인덱스를 Agentic Retrieval에서 사용할 수 있도록 래핑합니다.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeSource,
    SearchKnowledgeSourceData,
)

# Knowledge Source 정의
knowledge_source = KnowledgeSource(
    name=KNOWLEDGE_SOURCE_NAME,
    description="한국어 쇼핑몰 상품 데이터 (상품명, 브랜드, 카테고리, 리뷰 포함)",
    data=SearchKnowledgeSourceData(
        index_name=INDEX_NAME,
        semantic_configuration_name="semantic-config",
        vector_fields=["content_vector"],
        content_fields=["content"],
        title_field="title",
        keyword_fields=["brand", "category"]
    )
)

# Knowledge Source 생성 또는 업데이트
result = index_client.create_or_update_knowledge_source(knowledge_source)
print(f"✅ Knowledge Source '{result.name}' 생성/업데이트 완료")
print(f"   - 연결된 인덱스: {INDEX_NAME}")
print(f"   - 시맨틱 설정: semantic-config")

## 7. Knowledge Base 생성

**Knowledge Base**는 하나 이상의 Knowledge Source를 그룹화하고, LLM 기반 쿼리 플래닝을 구성합니다.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeSourceReference,
    KnowledgeAgentAzureOpenAIModel,
    KnowledgeRetrievalOutputMode,
    KnowledgeRetrievalMinimalReasoningEffort,
)

# LLM 모델 연결 설정
agent_model = KnowledgeAgentAzureOpenAIModel(
    azure_open_ai_resource=aoai_endpoint.replace("https://", "").split(".")[0],  # 리소스 이름만 추출
    azure_open_ai_deployment_id=chat_deployment,
    azure_open_ai_model_name=chat_deployment  # 모델 이름 (gpt-4o, gpt-4.1 등)
)

# Knowledge Base 정의
knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="쇼핑몰 상품 검색을 위한 지식 베이스",
    knowledge_sources=[
        KnowledgeSourceReference(name=KNOWLEDGE_SOURCE_NAME)
    ],
    models=[agent_model],
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,  # 원본 데이터 반환
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort()  # 기본값: minimal
)

# Knowledge Base 생성 또는 업데이트
result = index_client.create_or_update_knowledge_base(knowledge_base)
print(f"✅ Knowledge Base '{result.name}' 생성/업데이트 완료")
print(f"   - Knowledge Sources: {[ks.name for ks in result.knowledge_sources]}")
print(f"   - Output Mode: {result.output_mode}")
print(f"   - Reasoning Effort: {type(result.retrieval_reasoning_effort).__name__}")

## 8. 설정 확인

In [ ]:
# 생성된 리소스 확인
print("=" * 50)
print("📊 Agentic Retrieval 설정 요약")
print("=" * 50)

# 인덱스 정보
idx = index_client.get_index(INDEX_NAME)
print(f"\n📁 인덱스: {idx.name}")
print(f"   - 필드 수: {len(idx.fields)}")
print(f"   - 벡터 필드: content_vector")
print(f"   - 시맨틱 설정: {idx.semantic_search.default_configuration_name}")

# Knowledge Source 정보
ks = index_client.get_knowledge_source(KNOWLEDGE_SOURCE_NAME)
print(f"\n🔗 Knowledge Source: {ks.name}")
print(f"   - 설명: {ks.description}")

# Knowledge Base 정보
kb = index_client.get_knowledge_base(KNOWLEDGE_BASE_NAME)
print(f"\n🧠 Knowledge Base: {kb.name}")
print(f"   - 설명: {kb.description}")
print(f"   - 연결된 소스: {[s.name for s in kb.knowledge_sources]}")
print(f"   - 출력 모드: {kb.output_mode}")

print("\n" + "=" * 50)
print("✅ 설정 완료! 다음 노트북(02-agentic_retrieval.ipynb)에서 검색을 실행합니다.")
print("=" * 50)